# Fashion Florence V2 — LoRA Fine-Tuning (8-field tags)

Fine-tune **Florence-2** with LoRA on ~20k GPT-4o-mini labeled images to predict 8 structured fashion attributes.

**Base model:** `anushreeberlia/fashion-florence` (V1, trained on 50k images with 4 fields).

**New fields:** category, primary_color, secondary_colors, material, fit, style_tags, occasion_tags, season_tags.

**Requirements:** GPU runtime — go to *Runtime → Change runtime type → T4 GPU* (or A100 if available).

---
## Step 1 · Install Dependencies

In [ ]:
!pip install -q torch torchvision transformers==4.44.0 tokenizers==0.19.1 peft datasets pillow accelerate einops timm

---
## Step 2 · Mount Google Drive

Training data (`labeling_progress.jsonl`, images) and output (LoRA adapters) live on Drive so they survive runtime resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Verify data is accessible
!wc -l /content/drive/MyDrive/fashion-florence-v2/labeling_progress.jsonl

---
## Step 3 · Clone the Repo

In [ ]:
import os
os.chdir('/content')
!rm -rf fashion-florence
!git clone https://github.com/anushreeberlia/fashion-florence.git
os.chdir('/content/fashion-florence')

---
## Step 4 · Build Train / Val / Test Splits

Converts `labeling_progress.jsonl` (append-only, from GPT labeling) into the `train.jsonl`, `val.jsonl`, `test.jsonl` format that `train_lora.py` expects.

Default split: **90% train / 5% val / 5% test**.

You can re-run this step later with more labeled data and train again.

In [ ]:
!python training/build_splits_from_progress.py \
  /content/drive/MyDrive/fashion-florence-v2/labeling_progress.jsonl \
  --out-dir /content/drive/MyDrive/fashion-florence-v2

In [ ]:
# Sanity check: row counts and a sample row
!echo "Train:" && wc -l /content/drive/MyDrive/fashion-florence-v2/train.jsonl
!echo "Val:"   && wc -l /content/drive/MyDrive/fashion-florence-v2/val.jsonl
!echo "Test:"  && wc -l /content/drive/MyDrive/fashion-florence-v2/test.jsonl
!echo "\nSample row:" && head -1 /content/drive/MyDrive/fashion-florence-v2/train.jsonl | python -m json.tool

---
## Step 5 · Check GPU

Confirm we have a GPU and how much VRAM is available. This determines batch size.

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    vram = (getattr(props, 'total_memory', None) or getattr(props, 'total_mem', 0)) / 1e9
    print(f"GPU: {gpu}")
    print(f"VRAM: {vram:.1f} GB")
    if vram >= 35:
        print("→ Recommended: --per-device-batch-size 8 --grad-accum 2")
    elif vram >= 14:
        print("→ Recommended: --per-device-batch-size 4 --grad-accum 4")
    else:
        print("→ Recommended: --per-device-batch-size 2 --grad-accum 8")
else:
    print("ERROR: No GPU. Go to Runtime → Change runtime type → T4 GPU")

---
## Step 6 · Train

This runs LoRA fine-tuning on the V1 merged model (`anushreeberlia/fashion-florence`).

**What happens:**
1. Loads the V1 model + processor from HuggingFace.
2. Freezes all 230M original parameters.
3. Adds LoRA adapter matrices (~7M trainable params, ~3% of total).
4. Trains for 3 epochs on train split, evaluating on val split every 500 steps.
5. Saves the best checkpoint (lowest val loss) to Drive.

**Adjust `--per-device-batch-size` and `--grad-accum`** based on Step 5 output.

**Estimated time:** 1–3 hours depending on GPU and data size.

In [ ]:
!python training/train_lora.py \
  --train-file /content/drive/MyDrive/fashion-florence-v2/train.jsonl \
  --val-file /content/drive/MyDrive/fashion-florence-v2/val.jsonl \
  --output-dir /content/drive/MyDrive/fashion-florence-v2/lora_v2 \
  --model-id anushreeberlia/fashion-florence \
  --epochs 3 \
  --lr 2e-4 \
  --per-device-batch-size 4 \
  --grad-accum 4 \
  --fp16 \
  --save-steps 500 \
  --eval-steps 500 \
  --logging-steps 50

---
## Step 7 · Verify Training Output

Check that the LoRA adapter and processor were saved to Drive.

In [ ]:
import os
lora_dir = '/content/drive/MyDrive/fashion-florence-v2/lora_v2'
files = os.listdir(lora_dir)
print(f"Files in {lora_dir}:")
for f in sorted(files):
    size = os.path.getsize(os.path.join(lora_dir, f)) / 1e6
    print(f"  {f:40s} {size:>8.2f} MB")

---
## Step 8 · Test the Fine-Tuned Model

Load the LoRA adapter on top of the base model and run inference on a few test images.

In [ ]:
import json
import torch
from pathlib import Path
from PIL import Image
from transformers import AutoModelForCausalLM, AutoProcessor
from peft import PeftModel

BASE_MODEL = 'anushreeberlia/fashion-florence'
LORA_DIR = '/content/drive/MyDrive/fashion-florence-v2/lora_v2'

device = 'cuda' if torch.cuda.is_available() else 'cpu'

processor = AutoProcessor.from_pretrained(LORA_DIR, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, trust_remote_code=True)
model = PeftModel.from_pretrained(base_model, LORA_DIR)
model = model.to(device)
model.eval()

print(f'Model loaded on {device}')

In [ ]:
# Load a few test images and run inference
test_file = '/content/drive/MyDrive/fashion-florence-v2/test.jsonl'
test_rows = [json.loads(line) for line in open(test_file)][:5]

PROMPT = 'Analyze this clothing item image and return structured fashion tags as JSON.'

for i, row in enumerate(test_rows):
    img_path = row['image_path']
    if not os.path.isabs(img_path):
        img_path = os.path.join('/content/fashion-florence', img_path)
    
    image = Image.open(img_path).convert('RGB')
    inputs = processor(text=PROMPT, images=image, return_tensors='pt').to(device)
    
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=3,
            early_stopping=True,
        )
    decoded = processor.tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    
    print(f'\n--- Test image {i+1} ---')
    print(f'Path: {row["image_path"]}')
    print(f'Expected: {row["target"]}')
    print(f'Predicted: {decoded}')
    
    try:
        pred = json.loads(decoded)
        expected = json.loads(row['target'])
        match = pred.get('category') == expected.get('category')
        print(f'Category match: {match}')
    except json.JSONDecodeError:
        print('(Could not parse prediction as JSON)')

---
## Step 9 · Merge LoRA into Base Model (optional)

Merge the adapter weights into the base model for faster inference (no PEFT overhead).

Do this when you're happy with the model and ready to deploy.

In [ ]:
MERGED_DIR = '/content/drive/MyDrive/fashion-florence-v2/merged_v2'

merged = model.merge_and_unload()
merged.save_pretrained(MERGED_DIR)
processor.save_pretrained(MERGED_DIR)

print(f'Merged model saved to {MERGED_DIR}')
!ls -lh {MERGED_DIR}/

---
## Step 10 · Push to HuggingFace Hub (optional)

Upload the merged model so it can be used via the HF Inference API.

In [ ]:
# Uncomment and fill in when ready:

# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_folder(
#     folder_path=MERGED_DIR,
#     repo_id='anushreeberlia/fashion-florence',  # or a new repo
#     repo_type='model',
#     commit_message='V2: 8-field fashion tags, LoRA on 20k GPT-4o-mini labels',
# )
# print('Pushed to HuggingFace!')

---
## What to Do When You Have More Data

1. Resume `generate_gpt_labels.py` to grow `labeling_progress.jsonl` (e.g. to 30k).
2. Re-run **Step 4** to rebuild splits from the updated progress file.
3. Re-run **Step 6** with a new `--output-dir` (e.g. `lora_30k`).
4. Compare val loss between runs to see if more data helped.